- Github repository
https://github.com/AkaliKong/MiniOneRec?tab=readme-ov-file

- Transformers 默认下载模型路径： ~/.cache/huggingface/hub/

In [1]:
!ls

LICENSE			data		       rl.sh
LogitProcessor.py	data.py		       rl_gpr.py
README.md		data_test.py	       rq
SASRecModules_ori.py	evaluate.py	       run_mini_onerec_demo.ipynb
assets			evaluate.sh	       sasrec.py
calc.py			installed.txt	       sft.py
config			merge.py	       sft.sh
convert_dataset.py	minionerec_trainer.py  sft_gpr.py
convert_dataset.sh	requirements.txt       split.py
convert_dataset_gpr.py	rl.py		       utility.py


In [10]:
!python ./data/amazon18_data_process.py \
    --reviews_file ./data/review-Alabama_10.json \
    --user_k 5 \
    --item_k 5 \
    --st_year 1996 \
    --st_month 10 \
    --ed_year 2018 \
    --ed_month 10 \
    --output_path ./data/Amazon10

Processing dataset: Arts
Initial time range: 1996-10 to 2018-10
K-core threshold: 5
Loading reviews...
Loaded 5146330 total reviews
Metadata file ../meta_Arts.json not found
Error: No metadata found for dataset Arts
Failed to process dataset


In [26]:
# download from https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/

# wget https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/metaFiles2/meta_Arts_Crafts_and_Sewing.json.gz
# wget https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/categoryFiles/Arts_Crafts_and_Sewing.json.gz

!python ./data/amazon18_data_process.py \
    --reviews_file  ./data/Arts_Crafts_and_Sewing.json \
    --metadata_file ./data/meta_Arts_Crafts_and_Sewing.json \
    --user_k 5 \
    --item_k 5 \
    --st_year 1996 \
    --st_month 10 \
    --ed_year 2018 \
    --ed_month 10 \
    --output_path ./data/Arts_Crafts_and_Sewing

Processing dataset: Arts
Initial time range: 1996-10 to 2018-10
K-core threshold: 5
Loading reviews...
Loaded 2875917 total reviews
Processing metadata: 100%|██████████| 302988/302988 [00:00<00:00, 736299.24it/s]
Loaded 302988 metadata items, 284491 with valid titles
Performing k-core filtering...
K-core filtering: 100%|███████████| 2875917/2875917 [00:24<00:00, 115091.27it/s]
Users: 1411829, Items: 284483, Reviews: 2560805, Density: 6.375850005934098e-06
K-core filtering: 100%|██████████| 2560805/2560805 [00:01<00:00, 1588525.99it/s]
Users: 76407, Items: 58442, Reviews: 602760, Density: 0.0001349852161022948
K-core filtering: 100%|█████████████| 602760/602760 [00:00<00:00, 945447.12it/s]
Users: 62297, Items: 24891, Reviews: 489469, Density: 0.0003156572099431132
K-core filtering: 100%|█████████████| 489469/489469 [00:00<00:00, 823062.71it/s]
Users: 51089, Items: 22620, Reviews: 442199, Density: 0.0003826465072593364
K-core filtering: 100%|█████████████| 442199/442199 [00:00<00:00, 958

# Encode text to embedding
using Qwen text embedding to convert text to emb

In [35]:
# !sh ./rq/text2emb/amazon_text2emb.sh \
#      --dataset Arts_Crafts_and_Sewing \
#      --root ./data/Arts_Crafts_and_Sewing \
#      --plm_name Qwen/Qwen3-Embedding-0.6B

!accelerate launch --num_processes 8 ./rq/text2emb/amazon_text2emb.py \
    --dataset Appliances \
    --root ./data/Appliances/Appliances/ \
    --plm_checkpoint Qwen/Qwen3-Embedding-0.6B

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Running with 1 processes.
Process text data: 
Dataset:  Appliances
args.root:  ./data/Appliances/Appliances/
Loading Qwen Model: Qwen/Qwen3-Embedding-0.6B
tokenizer_config.json: 9.71kB [00:00, 23.1MB/s]
vocab.json: 2.78MB [00:01, 1.49MB/s]
merges.txt: 1.67MB [00:00, 1.83MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:03<00:00, 3.38MB/s]
config.json: 727B [00:00, 4.48MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|███████████████████| 1.19G/1.19G [05:00<00:00, 3.96MB/s]
Total items: 4
Start generating embeddings with 1 processes...
Proc 0: 100%|█████████████████████████████████████| 4/4 [00:00<00:00,  5.0

In [51]:
!accelerate launch --num_processes 8 ./rq/text2emb/amazon_text2emb.py \
    --dataset Arts \
    --root ./data/Arts/Arts/ \
    --max_sent_len 512 \
    --plm_checkpoint bert-large-cased
    # --plm_checkpoint Qwen/Qwen3-Embedding-0.6B

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Running with 1 processes.
Process text data: 
Dataset:  Arts
args.root:  ./data/Arts/Arts/
Loading Qwen Model: bert-large-cased
`torch_dtype` is deprecated! Use `dtype` instead!
Total items: 19471
Start generating embeddings with 1 processes...
Proc 0: 100%|███████████████████████████| 19471/19471 [1:18:17<00:00,  4.15it/s]
Gathering finished. Sorting and saving...
Final Embeddings shape:  (19471, 1024)
Saved to ./data/Arts/Arts/Arts.emb-qwen-td.npy


In [2]:
!du -h ./data/Arts/Arts/Arts.emb-qwen-td.npy

77M	./data/Arts/Arts/Arts.emb-qwen-td.npy


# Train RQ K-Means 

In [28]:
!pwd

/home/wwk/workspace/ai_project/AlgorithmCodingPractice/generative_recommendation/MiniOneRec


In [32]:
!python ./rq/rqkmeans_faiss.py --dataset Arts --data_path ./data/Arts/Arts/Arts.emb-qwen-td.npy  --output_root ./data/

loading: ./data/Arts/Arts/Arts.emb-qwen-td.npy
shape: (19471, 1024)
Training FAISS ResidualQuantizer
  data=19471  dim=1024  levels=3  codebook=256  total_codes=16,777,216
  training completed

Encoding 19471 vectors ...
  done, codes.shape=(19471, 3)

Before balancing:
  total=19471
  L1: unique=256
  L2: unique=256
  L3: unique=256
  unique full-paths=15592  collision_rate=0.1992
Saved indices: ./data/Arts/Arts.faiss-rq.index.json
Saved faiss quantizer: ./data/Arts/Arts.faiss-rq.index.faiss


In [35]:
# !python ./rq/generate_indices.py

!mkdir -p ./data/Arts/index
!cp  ./data/Arts/Arts/Arts.faiss-rq.index.faiss ./data/Arts/Arts/Arts.index.faiss
!cp  ./data/Arts/Arts/Arts.faiss-rq.index.json ./data/Arts/Arts/Arts.index.json
!ls ./data/Arts/Arts

Arts.emb-qwen-td.npy	   Arts.index.json  Arts.review.json  Arts.valid.inter
Arts.faiss-rq.index.faiss  Arts.inter.json  Arts.test.inter
Arts.faiss-rq.index.json   Arts.item.json   Arts.train.inter
Arts.index.faiss	   Arts.item2id     Arts.user2id


In [36]:
!python convert_dataset.py \
     --dataset_name Arts \
     --data_dir ./data/Arts/Arts/ \
     --output_dir ./data/Arts/


Loading Arts data from ./data/Arts/Arts/
Found 19471 items
Found 19471 item-to-semantic mappings
  Item 0: ['<a_90>', '<b_38>', '<c_181>'] -> <a_90><b_38><c_181>
  Item 1: ['<a_201>', '<b_82>', '<c_190>'] -> <a_201><b_82><c_190>
  Item 2: ['<a_235>', '<b_211>', '<c_61>'] -> <a_235><b_211><c_61>
Created item info file: ./data/Arts/info/Arts_5_2016-10-2018-11.txt
Created train file: ./data/Arts/train/Arts_5_2016-10-2018-11.csv with 295625 rows
  Kept all sequences for train data
  Sample row:
    user_id: A900
    history_item_id: [4920]
    item_id: 4920
    history_item_sid: ['<a_4><b_211><c_104>']
    item_sid: <a_4><b_211><c_104>
    item_title: Richard Simmons Foodmover...
Created valid file: ./data/Arts/valid/Arts_5_2016-10-2018-11.csv with 36953 rows
  Sample row:
    user_id: A35472
    history_item_id: [4212, 205, 17484, 239, 197, 323]
    item_id: 3837
    history_item_sid: ['<a_209><b_208><c_94>', '<a_75><b_76><c_114>', '<a_163><b_58><c_9>', '<a_117><b_135><c_0>', '<a_33><b_14

In [41]:
!ls ./data/Arts/

Arts			   Arts.faiss-rq.index.json  info    test   valid
Arts.faiss-rq.index.faiss  index		     output  train


In [43]:
# !rm ./data/Arts/Arts.faiss-rq.index.faiss ./data/Arts/Arts.faiss-rq.index.json

In [55]:
!head -n 3 ./data/Arts/train/Arts_5_2016-10-2018-11.csv

user_id,history_item_title,item_title,history_item_id,item_id,history_item_sid,item_sid
A900,['Richard Simmons Foodmover'],Richard Simmons Foodmover,[4920],4920,['<a_4><b_211><c_104>'],<a_4><b_211><c_104>
A900,"['Richard Simmons Foodmover', 'Richard Simmons Foodmover']",Richard Simmons Foodmover,"[4920, 4920]",4920,"['<a_4><b_211><c_104>', '<a_4><b_211><c_104>']",<a_4><b_211><c_104>


In [57]:
import pandas as pd

df = pd.read_csv("./data/Arts/train/Arts_5_2016-10-2018-11.csv")
df.head(10)

,user_id,history_item_title,item_title,history_item_id,item_id,history_item_sid,item_sid
0,A900,['Richard Simmons Foodmover'],Richard Simmons Foodmover,[4920],4920,['<a_4><b_211><c_104>'],<a_4><b_211><c_104>
1,A900,"['Richard Simmons Foodmover', 'Richard Simmons...",Richard Simmons Foodmover,"[4920, 4920]",4920,"['<a_4><b_211><c_104>', '<a_4><b_211><c_104>']",<a_4><b_211><c_104>
2,A4956,['Fiskars Basic Shapes Ultra ShapeXpress Start...,Xyron 23675 Adhesive 2 Inch by 2 Inch Eraser,[9367],4710,['<a_191><b_91><c_156>'],<a_159><b_211><c_27>
3,A4956,['Fiskars Basic Shapes Ultra ShapeXpress Start...,"EK Success Herma Dotto Permanent Refill"" />","[9367, 4710]",3611,"['<a_191><b_91><c_156>', '<a_159><b_211><c_27>']",<a_159><b_49><c_25>
4,A4956,['Fiskars Basic Shapes Ultra ShapeXpress Start...,Xyron 23675 Adhesive 2 Inch by 2 Inch Eraser,"[9367, 4710, 3611]",4710,"['<a_191><b_91><c_156>', '<a_159><b_211><c_27>...",<a_159><b_211><c_27>
5,A4956,['Fiskars Basic Shapes Ultra ShapeXpress Start...,"EK Success Herma Dotto Permanent Refill"" />","[9367, 4710, 3611, 4710]",3611,"['<a_191><b_91><c_156>', '<a_159><b_211><c_27>...",<a_159><b_49><c_25>
6,A900,"['Richard Simmons Foodmover', 'Richard Simmons...",Richard Simmons Foodmover,"[4920, 4920, 4920]",4920,"['<a_4><b_211><c_104>', '<a_4><b_211><c_104>',...",<a_4><b_211><c_104>
7,A900,"['Richard Simmons Foodmover', 'Richard Simmons...",Richard Simmons Foodmover,"[4920, 4920, 4920, 4920]",4920,"['<a_4><b_211><c_104>', '<a_4><b_211><c_104>',...",<a_4><b_211><c_104>
8,A28041,"['Rit All-Purpose Powder Dye, Navy Blue']","Rit Dye 0340179 Powder-Color Remover, by The Yard",[8582],702,['<a_197><b_214><c_101>'],<a_12><b_82><c_101>
9,A28041,"['Rit All-Purpose Powder Dye, Navy Blue', 'Rit...",Crayola 8-Count Fabric Crayons,"[8582, 702]",5163,"['<a_197><b_214><c_101>', '<a_12><b_82><c_101>']",<a_18><b_103><c_61>


In [64]:
len(df),len(df.sample(frac=0.1))

(295625, 29562)

In [59]:
!ls ./data/Arts/Arts/Arts.index.json
!head -n 10 ./data/Arts/Arts/Arts.item.json

./data/Arts/Arts/Arts.index.json
{
  "0": {
    "title": "Loew-Cornell Simply Art Wood Jumbo Craft Sticks 300 ct.",
    "description": "['SIMPLY ART Wood crafting products make crafting fun and come in a wide variety of sizes and shapes, from teardrops, circles, and hearts to traditional craft favorites like craft sticks and clothespins. Loew-Cornells SIMPLY ART Wood crafting products are the leading name in wood crafting and is the perfect addition to any craft set.']",
    "brand": "Loew-Cornell",
    "categories": ""
  },
  "1": {
    "title": "Singer 2131E All Purpose Machine Oil",
    "description": "['Use SINGER All Purpose Machine Oil to keep your sewing machine running smoothly. The oil eliminates friction between touching metal parts and prevents damage caused by abrasion and rust. The specially formulated lubricant can be used on sewing/knitting machines, vacuum cleaners, household appliances, typewriters, computers, bicycle gears and other appliances. Comes in a 4-ounce bott

# SFT

In [75]:
%%writefile sft_arts.sh
export NCCL_IB_DISABLE=1        # 完全禁用 IB/RoCE
# Office_Products, Industrial_and_Scientific
# 这里使用sample=40 40个样本 进行采样数据缩短训练时间
for category in "Arts"; do
    train_file=$(ls -f ./data/Arts/train/${category}*11.csv)
    eval_file=$(ls -f ./data/Arts/valid/${category}*11.csv)
    test_file=$(ls -f ./data/Arts/test/${category}*11.csv)
    info_file=$(ls -f ./data/Arts/info/${category}*.txt)
    echo ${train_file} ${eval_file} ${info_file} ${test_file}
    
    torchrun --nproc_per_node 1 \
            sft.py \
            --base_model Qwen/Qwen2.5-0.5B \
            --batch_size 128 \
            --micro_batch_size 16 \
            --train_file ${train_file} \
            --eval_file ${eval_file} \
            --output_dir ./data/model_qwen2.5 \
            --wandb_project wandb_proj \
            --wandb_run_name wandb_name \
            --category ${category} \
            --train_from_scratch False \
            --seed 42 \
            --sample 40 \
            --sid_index_path ./data/Arts/Arts/Arts.index.json \
            --item_meta_path ./data/Arts/Arts/Arts.item.json \
            --freeze_LLM False
done




Overwriting sft_arts.sh


In [51]:
!sh sft_arts.sh \
     --base_model Qwen/Qwen2.5-0.5B \
     --output_dir ./data/model_qwen2.5 \
     --sid_index_path ./data/Arts/Arts/Arts.index.json \
     --item_meta_path ./data/Arts/Arts/Arts.item.json

./data/Arts/train/Arts_5_2016-10-2018-11.csv ./data/Arts/valid/Arts_5_2016-10-2018-11.csv ./data/Arts/info/Arts_5_2016-10-2018-11.txt ./data/Arts/test/Arts_5_2016-10-2018-11.csv
Arts
`torch_dtype` is deprecated! Use `dtype` instead!
Loading index from ./data/Arts/Arts/Arts.index.json
Adding 768 new tokens to tokenizer
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
  0%|                                                | 0/295625 [00:00<?, ?it/s]/home/wwk/workspace/ai_project/AlgorithmCodingPractice/generative_recommendation/MiniOneRec/data.py:404: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  row['his

In [76]:
!sh sft_arts.sh \
     --base_model Qwen/Qwen2.5-0.5B \
     --output_dir ./data/model_qwen2.5 \
     --sid_index_path ./data/Arts/Arts/Arts.index.json \
     --item_meta_path ./data/Arts/Arts/Arts.item.json

./data/Arts/train/Arts_5_2016-10-2018-11.csv ./data/Arts/valid/Arts_5_2016-10-2018-11.csv ./data/Arts/info/Arts_5_2016-10-2018-11.txt ./data/Arts/test/Arts_5_2016-10-2018-11.csv
Arts
`torch_dtype` is deprecated! Use `dtype` instead!
Loading index from ./data/Arts/Arts/Arts.index.json
Adding 768 new tokens to tokenizer
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
  0%|                                                    | 0/40 [00:00<?, ?it/s]/home/wwk/workspace/ai_project/AlgorithmCodingPractice/generative_recommendation/MiniOneRec/data.py:404: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  row['his

# RL 

In [85]:
%%writefile rl_arts.sh
#!/bin/bash

export NCCL_IB_DISABLE=1        # 完全禁用 IB/RoCE

for category in "Arts"; do
    train_file=$(ls -f ./data/Arts/train/${category}*.csv)
    eval_file=$(ls -f ./data/Arts/valid/${category}*11.csv)
    info_file=$(ls -f ./data/Arts/info/${category}*.txt)

    HF_ENDPOINT=https://hf-mirror.com accelerate launch \
                                    --config_file ./config/zero2_opt.yaml \
                                    --num_processes 1 --main_process_port 29503 \
                                    rl.py \
                        --model_path ./data/model_qwen2.5/ \
                        --sample 5 \
                        --train_batch_size 64 \
                        --eval_batch_size 128 \
                        --num_train_epochs 2 \
                        --gradient_accumulation_steps 2 \
                        --train_file ${train_file} \
                        --eval_file ${eval_file} \
                        --info_file ${info_file} \
                        --category ${category} \
                        --sample_train True \
                        --eval_step 0.0999 \
                        --reward_type ranking \
                        --num_generations 16 \
                        --mask_all_zero False \
                        --dynamic_sampling False \
                        --sync_ref_model True \
                        --beam_search True \
                        --test_during_training False \
                        --temperature 1.0 \
                        --learning_rate 1e-5 \
                        --add_gt False \
                        --beta 1e-3 \
                        --dapo False \
                        --output_dir ./data/rl_model_qwen2.5 \
                        --wandb_run_name wandb_name \
                        --sid_index_path ./data/Arts/Arts/Arts.index.json \
                        --item_meta_path ./data/Arts/Arts/Arts.item.json
done


Overwriting rl_arts.sh


In [87]:
!sh rl_arts.sh

/home/wwk/workspace/miniconda3/envs/llm_study/lib/python3.13/site-packages/trl/import_utils.py:91: UserWarning: TRL currently only supports vLLM version `0.10.2`. You have version 0.11.2 installed. We recommend to install this version to avoid compatibility issues.
  warnings.warn(
/home/wwk/workspace/miniconda3/envs/llm_study/lib/python3.13/site-packages/trl/import_utils.py:91: UserWarning: TRL currently only supports vLLM version `0.10.2`. You have version 0.11.2 installed. We recommend to install this version to avoid compatibility issues.
  warnings.warn(
Arts
  0%|                                                     | 0/5 [00:00<?, ?it/s]/home/wwk/workspace/ai_project/AlgorithmCodingPractice/generative_recommendation/MiniOneRec/data.py:363: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  row['histor

# Eval model

In [94]:
%%writefile eval_arts.sh
# Industrial_and_Scientific
# Office_Products
for category in "Arts"
do
    # your model path 
    # 填写base_model 版本
    exp_name="Qwen/Qwen2.5-0.5B"

    exp_name_clean=$(basename "$exp_name")
    echo "Processing category: $category with model: $exp_name_clean (STANDARD MODE)"
    
    train_file=$(ls ./data/Arts/train/${category}*.csv 2>/dev/null | head -1)
    test_file=$(ls ./data/Arts/test/${category}*11.csv 2>/dev/null | head -1)
    info_file=$(ls ./data/Arts/info/${category}*.txt 2>/dev/null | head -1)
    
    if [[ ! -f "$test_file" ]]; then
        echo "Error: Test file not found for category $category"
        continue
    fi
    if [[ ! -f "$info_file" ]]; then
        echo "Error: Info file not found for category $category"
        continue
    fi
    
    temp_dir="./temp/${category}-${exp_name_clean}"
    echo "Creating temp directory: $temp_dir"
    mkdir -p "$temp_dir"
    
    echo "Splitting test data..."
    python ./split.py --input_path "$test_file" --output_path "$temp_dir" --cuda_list "0,1,2,3,4,5,6,7"
    
    if [[ ! -f "$temp_dir/0.csv" ]]; then
        echo "Error: Data splitting failed for category $category"
        continue
    fi
    
    cudalist="0 1 2 3 4 5 6 7"  
    echo "Starting parallel evaluation (STANDARD MODE)..."
    for i in ${cudalist}
    do
        if [[ -f "$temp_dir/${i}.csv" ]]; then
            echo "Starting evaluation on GPU $i for category ${category}"
            CUDA_VISIBLE_DEVICES=$i python -u ./evaluate.py \
                --base_model "$exp_name" \
                --sample 4 \
                --info_file "$info_file" \
                --category ${category} \
                --test_data_path "$temp_dir/${i}.csv" \
                --result_json_data "$temp_dir/${i}.json" \
                --batch_size 8 \
                --num_beams 50 \
                --max_new_tokens 256 \
                --temperature 1.0 \
                --guidance_scale 1.0 \
                --length_penalty 0.0 &
        else
            echo "Warning: Split file $temp_dir/${i}.csv not found, skipping GPU $i"
        fi
    done
    echo "Waiting for all evaluation processes to complete..."
    wait
    
    result_files=$(ls "$temp_dir"/*.json 2>/dev/null | wc -l)
    if [[ $result_files -eq 0 ]]; then
        echo "Error: No result files generated for category $category"
        continue
    fi
    
    output_dir="./results/${exp_name_clean}"
    echo "Creating output directory: $output_dir"
    mkdir -p "$output_dir"

    actual_cuda_list=$(ls "$temp_dir"/*.json 2>/dev/null | sed 's/.*\///g' | sed 's/\.json//g' | tr '\n' ',' | sed 's/,$//')
    echo "Merging results from GPUs: $actual_cuda_list"
    
    python ./merge.py \
        --input_path "$temp_dir" \
        --output_path "$output_dir/final_result_${category}.json" \
        --cuda_list "$actual_cuda_list"
    
    if [[ ! -f "$output_dir/final_result_${category}.json" ]]; then
        echo "Error: Result merging failed for category $category"
        continue
    fi
    
    echo "Calculating metrics..."
    python ./calc.py \
        --path "$output_dir/final_result_${category}.json" \
        --item_path "$info_file"
    
    echo "Completed processing for category: $category"
    echo "Results saved to: $output_dir/final_result_${category}.json"
    echo "----------------------------------------" 
done

echo "All categories processed!"



Overwriting eval_arts.sh


In [95]:
!bash eval_arts.sh

Processing category: Arts with model: Qwen2.5-0.5B (STANDARD MODE)
Creating temp directory: ./temp/Arts-Qwen2.5-0.5B
Splitting test data...
Starting parallel evaluation (STANDARD MODE)...
Starting evaluation on GPU 0 for category Arts
Starting evaluation on GPU 1 for category Arts
Starting evaluation on GPU 2 for category Arts
Starting evaluation on GPU 3 for category Arts
Starting evaluation on GPU 4 for category Arts
Starting evaluation on GPU 5 for category Arts
Starting evaluation on GPU 6 for category Arts
Starting evaluation on GPU 7 for category Arts
Waiting for all evaluation processes to complete...
free(): double free detected in tcache 2
free(): double free detected in tcache 2
free(): double free detected in tcache 2
free(): double free detected in tcache 2
free(): double free detected in tcache 2
free(): double free detected in tcache 2
free(): double free detected in tcache 2
arts
`torch_dtype` is deprecated! Use `dtype` instead!
  0%|                                     